In [3]:
import pandas as pd
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import BayesianEstimator
from pgmpy.inference import VariableElimination

# Load dataset
data = pd.read_csv("heart.csv")

# Select columns
data = data[['age', 'sex', 'cp', 'exang', 'thalach', 'target']]

# Convert age into categories
data['age'] = pd.cut(data['age'],
                     bins=[0, 40, 60, 100],
                     labels=[0, 1, 2]).astype(int)

# Convert heart rate into categories
data['thalach'] = pd.cut(data['thalach'],
                         bins=[0, 100, 150, 250],
                         labels=[0, 1, 2]).astype(int)

# Create Bayesian Network
model = DiscreteBayesianNetwork([
    ('age', 'target'),
    ('sex', 'target'),
    ('cp', 'target'),
    ('exang', 'target'),
    ('thalach', 'target')
])

# Train the model
estimator = BayesianEstimator(model, data)
cpds = estimator.get_parameters(prior_type='BDeu')
model.add_cpds(*cpds)

print("Model trained successfully!")

# Check model
print("Model Valid:", model.check_model())

# Perform inference
inference = VariableElimination(model)

result = inference.query(
    variables=['target'],
    evidence={
        'age': 1,
        'sex': 1,
        'cp': 2,
        'exang': 0,
        'thalach': 2
    }
)

# Display output
print("\nHeart Disease Diagnosis:")
print(result)

if result.values[1] > result.values[0]:
    print("Heart Disease Detected")
else:
    print("No Heart Disease Detected")

Model trained successfully!
Model Valid: True

Heart Disease Diagnosis:
+-----------+---------------+
| target    |   phi(target) |
+===========+===============+
| target(0) |        0.9915 |
+-----------+---------------+
| target(1) |        0.0085 |
+-----------+---------------+
No Heart Disease Detected


C:\Users\Hp\AppData\Local\Temp\ipykernel_5268\3372343821.py:32: FutureWarning: `pgmpy.estimators.BayesianEstimator` is deprecated and will be removed in v1.3.0. Please use `pgmpy.parameter_estimator.DiscreteBayesianEstimator` instead.
  estimator = BayesianEstimator(model, data)
